In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-lgd-get-n-feats'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import json
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '03_pricing_lgd'
    
    # features to force into the model
    list_feats_force = [
        #'fltdowncash__app',
        'fltadvance__app',
        'ENG-loan_to_value',
        #'fltapproveddowntotal__app',
        #'payment__app',
        #'pti__app',
    ]
    
    # load output from concat sensitivity
    print('Loading output from sensitivity analysis concatenation...')
    str_filename = 'df_sensitivity.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/05_lambda_concat_sensitivity/{str_filename}'
    df = pd.read_csv(str_uri)
    
    # tag if it is a forced feature
    print('Tagging forced features and removing them...')
    df['force'] = df['feature'].apply(
        lambda x: 1 if x in list_feats_force else 0,
    )
    # remove forced features
    df = df[df['force'] == 0].copy()
    
    # subset to features that helped if removed
    print('Subsetting to just those features that helped the model if removed...')
    df = df[df['helped'] == 1].copy()
    
    # get the number of features
    print('Getting the number of features that helped the model once removed...')
    int_n_feats_helped = df.shape[0]
    
    # write to s3 as json for map in step function
    print('Writing json of number of features in model to s3...')
    str_n_feats_helped = json.dumps(int_n_feats_helped)
    cls_client_s3 = boto3.client('s3')
    str_filename = 'json_n_cols_in_model.json'
    str_key = f'{str_model}/02_model/02_model/06_lambda_get_n_feats_for_choice/{str_filename}'
    cls_client_s3.put_object(
        Bucket=str_project,
        Key=str_key,
        Body=str_n_feats_helped,
    )
    
    # return int_n_feats_helped
    return str_n_feats_helped

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-lgd-get-n-feats

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  37.89kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 303d20c679a3
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d25abc48fcf6
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> 3146759a8e36
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 7428a9a82206
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 7d27b5c11572
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 30a946b0e5f3
Removing intermediate container 30a946b0e5f3
 ---> dfd982ea426f
Successfully built dfd982ea426f
Successfully tagged genxii-lgd-get-n-feats:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-lgd-get-n-feats' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-get-n-feats]
f5124b55461d: Preparing
7259da6fe633: Preparing
e4a69b79c430: Preparing
d630e2305053: Preparing
3bd433acfe84: Preparing
09b55d38856d: Preparing
e073f5919ae5: Preparing
b3b414f01759: Preparing
8308f08f35ba: Preparing
c8203e562a8c: Preparing
09b55d38856d: Waiting
e073f5919ae5: Waiting
b3b414f01759: Waiting
8308f08f35ba: Waiting
c8203e562a8c: Waiting
f5124b55461d: Pushed
e4a69b79c430: Pushed
d630e2305053: Pushed
e073f5919ae5: Pushed
b3b414f01759: Pushed
8308f08f35ba: Pushed
3bd433acfe84: Pushed
7259da6fe633: Pushed
09b55d38856d: Pushed
c8203e562a8c: Pushed
latest: digest: sha256:0e86fa4787ee4e77441a30ff28083058983deb1921f6fdd43b433c69021b5bcc size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:09:36 GMT',
                                      'x-amzn-requestid': '79ee0bee-e5e4-4b77-9d44-513e7a30cd71'},
                      'HTTPStatusCode': 204,
                      'RequestId': '79ee0bee-e5e4-4b77-9d44-513e7a30cd71',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '0e86fa4787ee4e77441a30ff28083058983deb1921f6fdd43b433c69021b5bcc',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-get-n-feats',
 'FunctionName': 'genxii-lgd-get-n-feats',
 'LastModified': '2024-08-20T16:09:36.596+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-lgd-get-n-feats'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1198',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:09:37 GMT',
                                      'x-amzn-requestid': '1dae9e98-71f6-47be-9d3c-bd8f9c39674b'},
                      'HTTPStatusCode': 201,
                      'RequestId': '1dae9e98-71f6-47

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)